# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [4]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [5]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [6]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [7]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [8]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}] #We have to make sure to include the system and history in the messages we send to the API
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message #This is the message that contains the tool call information
        print(f"Tool call detected: {message}")
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [10]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [12]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool call detected: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ZFI44eJ2xUAowjZ8GvqZeYFC', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function')])
Tool called for city Tokyo
Tool call detected: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_57F1D09XmJATRMT0oQmrsgET', function=Function(arguments='{"destination_city":"berlin"}', name='get_ticket_price'), type='function')])
Tool called for city berlin
Tool call detected: ChatCompletionMessage(content='Let me check the ticket prices for both London and Berlin for you.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_bqKHVcTA0QI8hQ3AonSd94FI', func

Traceback (most recent call last):
  File "c:\Users\81908\miniconda3\envs\llms\Lib\site-packages\gradio\queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\81908\miniconda3\envs\llms\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\81908\miniconda3\envs\llms\Lib\site-packages\gradio\blocks.py", line 2162, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\81908\miniconda3\envs\llms\Lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\81908\miniconda3\envs\llms\Lib\site-packages\gradio\utils.py", line 1017, in async_wrapper
    response = await f(*args, **kwargs)
        

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [19]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools) #We include the tools in the API call so that the model can call them if needed

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [20]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [23]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Paris


In [22]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

Chỗ `while response.choices[0].finish_reason=="tool_calls":` trông có vẻ "ảo diệu", nhưng thực chất nó là cách giải quyết một kịch bản giao tiếp **rất thực tế**: AI cần phải gọi function **nhiều lần liên tiếp, kết quả của lần gọi hàm này quyết định hàm tiếp theo**.

Hãy cùng đi qua workflow chi tiết của nó:

### Setting kịch bản:
Giả sử bạn cung cấp cho AI 2 tools:
1. `get_customer_id(name)`: Tìm ID của khách hàng dựa vào tên.
2. `get_booking_info(customer_id)`: Tìm vé đã đặt dựa vào ID.

Ngươi dùng hỏi: *"Booking của tôi đâu? Tôi tên là Phuc Nguyen."*

### Workflow bước nhảy của vòng lặp `while`:

**Bước 1: Lần gọi API đầu tiên (Ngoài vòng lặp)**
```python
response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
```
- Bạn gửi câu hỏi *"Tôi tên là Phuc Nguyen"*.
- AI suy luận: *"Để lấy được booking, mình cần `customer_id`. Nhưng bây giờ mình chỉ có `name`. Vậy trước tiên mình cần gọi tool `get_customer_id("Phuc Nguyen")`."*
- AI trả về kết quả mang thông điệp: `finish_reason == "tool_calls"`.

---

**Bước 2: Vòng lặp `while` bắt đầu chạy (Lần 1)**
```python
while response.choices[0].finish_reason=="tool_calls":
```
- Điều kiện đúng (`finish_reason` đang là `tool_calls`). Nó nhảy vào trong `while`.
- Code của bạn trích xuất yêu cầu gọi hàm và chạy tool `get_customer_id("Phuc Nguyen")`.
- Tool trả về kết quả ví dụ là: `"ID-999"`.
- Code nối object `"role": "tool"` mang nội dung `"ID-999"` vào `messages`.
- Code **tiếp tục gọi API OpenAI lần nữa** ở cuối vòng `while`:
```python
response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
```
*(Lưu ý: Ngay lúc này, luồng code quay lại kiểm tra điều kiện `while`)*

---

**Bước 3: Vòng lặp `while` bắt đầu chạy (Lần 2 - ĐÂY LÀ CHỖ VẤN ĐỀ THỨ 2 GIẢI QUYẾT)*
- AI nhận được thông tin: `"ID-999"`. 
- Nó suy luận: *"À, có ID rồi. Bây giờ mình cần dùng ID này để tiếp tục gọi hàm `get_booking_info("ID-999")`."*
- Việc này tạo ra một `finish_reason` **MỚI**, và lại là `"tool_calls"`!

- Mệnh đề `while response.choices[0].finish_reason=="tool_calls":` **TIẾP TỤC ĐÚNG**.
*(Nếu ở đây bạn dùng `if` như code cũ, vòng lặp dừng luôn, code thoát khỏi hàm `chat` văng ra string rỗng cho gradio hiển thị lỗi)*.
- Nhờ có `while`, nó tiếp tục nhảy vào thân vòng lặp. Nó gọi `get_booking_info("ID-999")`.
- Tool chạy và trả về `"Vé đi Tokyo ngày 25/12"`.
- Code append dữ liệu này ghim vào `messages`, rồi lại gọi OpenAI lần 3:
```python
response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
```

---

**Bước 4: Thoát khỏi vòng lặp `while`**
- AI nhận đủ tất cả thông tin: *"Mình đã có tên, có ID, có vé."*
- AI lúc này không cần gọi hàm nào nữa. Nó bắt đầu sinh ra văn bản text tự nhiên: *"Chào bạn Phuc Nguyen, vé của bạn đi Tokyo vào ngày 25/12 đã được xác nhận nhé."*
- `response` trả về bây giờ có `finish_reason == "stop"` (nghĩa là đã trả lời xong bằng cách nói chuyện như bình thường).

- Luồng code dội lên xét lại điều kiện của `while`: `finish_reason` là `"stop"` -> **SAI** (`"stop" != "tool_calls"`).
- Vòng lặp `while` bị phá vỡ!
- Dòng lệnh cuối cùng chạy:
```python
return response.choices[0].message.content
```
Dòng lệnh trả ra câu text cuối cùng và Gradio hiển thị lên màn hình người dùng.

### Tóm tắt "độ ảo diệu":
Vòng lặp `while` đó giống như một cái **bàn bóng bàn**:
- AI ném bóng tới: *"Chạy giùm 1 function nhé"*.
- Code ném bóng trả lại: *"Kết quả đây, nói chuyện tiếp đi"*.
- Cứ thế đánh qua đánh lại, chừng nào AI bảo *"OK tui hết cần chạy function rồi, tui đã đủ thông tin để ghép thành câu văn text"*, thì lúc đó điều kiện `while` sai, ngừng lặp, và kết thúc chu trình chat.

In [1]:
import sqlite3


In [2]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [3]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [4]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [5]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [6]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [7]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>